In [0]:
WITH HourlyMetrics AS (
    SELECT 
        -- Standardizing time to the hour for trend lines
        DATE_TRUNC('hour', event_time) AS time_bucket,
        store_key,
        order_size,
        -- Metrics for visualization
        SUM(quantity) AS units,
        ROUND(SUM(revenue),2) AS hourly_revenue,
        ROUND(SUM(estimated_profit),2) AS hourly_profit,
        -- Calculating Unit Economics
        ROUND(AVG(unit_price),2) AS avg_price_point
    FROM maven_catalog.gold_schema.fact_kafka_orders
    WHERE revenue > 0 
      AND quantity > 0
    GROUP BY 1, 2, 3
)
SELECT 
    time_bucket,
    store_key,
    order_size,
    hourly_revenue,
    hourly_profit,
    avg_price_point,
    -- Profit Margin percentage for color-coding heatmaps
    ROUND((hourly_profit / NULLIF(hourly_revenue, 0)) * 100,2) AS margin_pct
FROM HourlyMetrics
WHERE hourly_revenue IS NOT NULL
ORDER BY time_bucket ASC;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT 
    -- Date and Time Grouping
    CAST(event_time AS DATE) AS order_date,
    EXTRACT(HOUR FROM event_time) AS order_hour,
    
    -- Categorical Dimensions
    store_key,
    order_size,
    
    -- Aggregated Metrics
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(quantity) AS total_units_sold,
    ROUND(SUM(revenue), 2) AS total_revenue,
    ROUND(SUM(estimated_profit), 2) AS total_profit,
    
    -- Calculated Performance Indicators
    ROUND(SUM(estimated_profit) / NULLIF(SUM(revenue), 0), 4) AS profit_margin
    
FROM maven_catalog.gold_schema.fact_kafka_orders
WHERE revenue > 0 
  AND quantity > 0
GROUP BY 1, 2, 3, 4
HAVING total_revenue > 0
ORDER BY order_date DESC, total_revenue DESC;

Databricks visualization. Run in Databricks to view.

Store Efficiency & Margin Analysis

In [0]:
SELECT 
    store_key,
    ROUND(SUM(quantity),2) AS total_units,
    ROUND(SUM(revenue),2) AS total_revenue,
    ROUND(SUM(estimated_profit),2) AS total_profit,
    -- Profit per unit sold
    ROUND(SUM(estimated_profit) / NULLIF(SUM(quantity), 0),2) AS profit_per_unit
FROM maven_catalog.gold_schema.fact_kafka_orders
WHERE revenue > 0
GROUP BY store_key
HAVING SUM(revenue) > 0;

Price Volatility & Outlier Detection

In [0]:
SELECT 
    product_key,
    unit_price,
    order_size,
    -- Ranking to find top products by frequency
    COUNT(*) OVER(PARTITION BY product_key) as sales_count
FROM maven_catalog.gold_schema.fact_kafka_orders
WHERE unit_price > 0
-- Focus on high-frequency products to see price distribution
QUALIFY sales_count > 1 
ORDER BY product_key;

Databricks visualization. Run in Databricks to view.

Intra-Day Order Velocity

In [0]:
SELECT 
    -- 1. TRUNCATE time so it can be grouped (The Fix)
    DATE_TRUNC('hour', event_time) AS time_axis, 
    store_key,
    order_size,
    
    -- 2. Performance KPIs (The Aggregates)
    ROUND(SUM(revenue),2) AS hourly_revenue,
    ROUND(SUM(estimated_profit),2) AS hourly_profit,
    
    -- 3. Operational Intensity KPI
    -- Revenue per unit to see price efficiency without a table
    ROUND(SUM(revenue) / NULLIF(SUM(quantity), 0),2) AS revenue_velocity

FROM maven_catalog.gold_schema.fact_kafka_orders
WHERE revenue > 0  -- Filters out 0 revenue records
  AND quantity > 0 -- Filters out 0 quantity records

-- 4. Every non-aggregated column MUST be here
GROUP BY 1, 2, 3 

-- 5. Final filter to ensure no empty/zero visual points
HAVING SUM(revenue) > 0 
ORDER BY time_axis DESC;